# TravelBuddy RAG

AI-powered travel planning assistant using Retrieval-Augmented Generation (RAG).

**Runs in Google Colab.**

## 1. Install libraries

In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters
!pip -q install sentence-transformers faiss-cpu transformers accelerate gradio


## 2. Import libraries

In [ ]:
import os
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer
import faiss

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("All libraries imported successfully!")


## 3. Create the travel knowledge base

In [ ]:
travel_data = {'Chennai': 'Destination: Chennai, Tamil Nadu, India\n\nChennai is a major coastal city in Tamil Nadu known for Marina Beach, temples, museums, shopping areas and South Indian food.\n\nPopular attractions:\n1. Marina Beach - famous long urban beach and good for sunrise and evening walks.\n2. Kapaleeshwarar Temple - historic Hindu temple in Mylapore.\n3. Fort St. George - important historical site.\n4. Government Museum - museum with archaeology, art and cultural collections.\n5. San Thome Basilica - famous church near the coast.\n6. Besant Nagar Beach - popular evening destination.\n7. DakshinaChitra - cultural museum showcasing South Indian heritage.\n\nRecommended duration: 2 to 3 days.\n\nBudget:\nBudget travelers can spend approximately ₹800 to ₹1,500 per day excluding accommodation.\nMid-range travelers can spend approximately ₹2,000 to ₹4,000 per day excluding accommodation.\n\nFood:\nPopular foods include dosa, idli, pongal, biryani, filter coffee, parotta and South Indian meals.\n\nBest time:\nNovember to February is generally more comfortable for sightseeing.\n\nTransportation:\nChennai has buses, suburban trains, metro rail, taxis and auto-rickshaws.\n\nSuggested itinerary:\nDay 1: Marina Beach, San Thome Basilica, Kapaleeshwarar Temple and Mylapore.\nDay 2: Fort St. George, Government Museum and shopping.\nDay 3: Besant Nagar Beach and DakshinaChitra.\n', 'Bangalore': 'Destination: Bengaluru, Karnataka, India\n\nBengaluru is known for technology companies, pleasant weather, gardens, cafes and historical attractions.\n\nPopular attractions:\n1. Lalbagh Botanical Garden.\n2. Cubbon Park.\n3. Bangalore Palace.\n4. Vidhana Soudha.\n5. ISKCON Temple.\n6. UB City.\n7. Nandi Hills, located outside the city.\n\nRecommended duration: 2 to 3 days.\n\nBudget:\nBudget travelers can spend approximately ₹900 to ₹1,600 per day excluding accommodation.\nMid-range travelers can spend approximately ₹2,000 to ₹4,000 per day excluding accommodation.\n\nFood:\nPopular choices include South Indian breakfast, biryani, street food, cafes and desserts.\n\nBest time:\nOctober to February is generally comfortable.\n\nTransportation:\nMetro, buses, taxis and auto-rickshaws are commonly used.\n\nSuggested itinerary:\nDay 1: Lalbagh, Vidhana Soudha and Cubbon Park.\nDay 2: Bangalore Palace, ISKCON Temple and local cafes.\nDay 3: Nandi Hills day trip.\n', 'Goa': 'Destination: Goa, India\n\nGoa is famous for beaches, Portuguese-influenced architecture, nightlife, seafood and relaxed vacations.\n\nPopular attractions:\n1. Baga Beach.\n2. Calangute Beach.\n3. Anjuna Beach.\n4. Vagator Beach.\n5. Fort Aguada.\n6. Basilica of Bom Jesus.\n7. Panjim.\n8. Dudhsagar Falls.\n\nRecommended duration: 3 to 5 days.\n\nBudget:\nBudget travelers can spend approximately ₹1,200 to ₹2,000 per day excluding accommodation.\nMid-range travelers can spend approximately ₹2,500 to ₹5,000 per day excluding accommodation.\n\nFood:\nGoan seafood, fish curry rice, vegetarian thalis and Portuguese-influenced dishes are popular.\n\nBest time:\nNovember to February is popular because of the generally pleasant weather.\n\nTransportation:\nRental scooters, taxis and buses are common options.\n\nSuggested itinerary:\nDay 1: Calangute, Baga and Anjuna.\nDay 2: Fort Aguada, Vagator and Panjim.\nDay 3: Old Goa churches and a relaxed beach evening.\nDay 4: Dudhsagar Falls or another nature activity.\n', 'Delhi': "Destination: New Delhi, India\n\nDelhi is known for historical monuments, markets, museums and diverse food.\n\nPopular attractions:\n1. India Gate.\n2. Red Fort.\n3. Qutub Minar.\n4. Humayun's Tomb.\n5. Lotus Temple.\n6. Akshardham Temple.\n7. Connaught Place.\n8. Chandni Chowk.\n\nRecommended duration: 3 days.\n\nBudget:\nBudget travelers can spend approximately ₹900 to ₹1,600 per day excluding accommodation.\nMid-range travelers can spend approximately ₹2,000 to ₹4,000 per day excluding accommodation.\n\nFood:\nDelhi is famous for chaat, parathas, kebabs, chole bhature and street food.\n\nBest time:\nOctober to March is generally preferred for sightseeing.\n\nTransportation:\nDelhi Metro is one of the most useful transportation options. Buses, taxis and auto-rickshaws are also available.\n\nSuggested itinerary:\nDay 1: India Gate, Humayun's Tomb and Lotus Temple.\nDay 2: Red Fort, Chandni Chowk and Jama Masjid area.\nDay 3: Qutub Minar, Connaught Place and local shopping.\n", 'Mumbai': "Destination: Mumbai, Maharashtra, India\n\nMumbai is India's major financial and entertainment center and is known for its coastline, architecture and street food.\n\nPopular attractions:\n1. Gateway of India.\n2. Marine Drive.\n3. Chhatrapati Shivaji Maharaj Terminus.\n4. Colaba.\n5. Elephanta Caves.\n6. Siddhivinayak Temple.\n7. Juhu Beach.\n\nRecommended duration: 2 to 3 days.\n\nBudget:\nBudget travelers can spend approximately ₹1,000 to ₹1,800 per day excluding accommodation.\nMid-range travelers can spend approximately ₹2,500 to ₹5,000 per day excluding accommodation.\n\nFood:\nVada pav, pav bhaji, misal pav, bhel puri and seafood are popular.\n\nBest time:\nNovember to February is generally comfortable.\n\nTransportation:\nLocal trains, metro, buses, taxis and auto-rickshaws are commonly used.\n\nSuggested itinerary:\nDay 1: Gateway of India, Colaba and Marine Drive.\nDay 2: CSMT, museums and Juhu Beach.\nDay 3: Elephanta Caves and nearby attractions.\n", 'Jaipur': 'Destination: Jaipur, Rajasthan, India\n\nJaipur is known as the Pink City and is famous for forts, palaces, markets and Rajasthani culture.\n\nPopular attractions:\n1. Amber Fort.\n2. City Palace.\n3. Hawa Mahal.\n4. Jantar Mantar.\n5. Jal Mahal.\n6. Nahargarh Fort.\n7. Albert Hall Museum.\n\nRecommended duration: 2 to 3 days.\n\nBudget:\nBudget travelers can spend approximately ₹900 to ₹1,500 per day excluding accommodation.\nMid-range travelers can spend approximately ₹2,000 to ₹4,000 per day excluding accommodation.\n\nFood:\nDal baati churma, ghewar, kachori, lassi and Rajasthani thali are popular.\n\nBest time:\nOctober to March is generally preferred.\n\nTransportation:\nAuto-rickshaws, taxis and buses are available.\n\nSuggested itinerary:\nDay 1: Amber Fort, Jal Mahal and City Palace.\nDay 2: Hawa Mahal, Jantar Mantar and local markets.\nDay 3: Nahargarh Fort and Albert Hall Museum.\n', 'Kerala': 'Destination: Kerala, India\n\nKerala is known for beaches, backwaters, hill stations, wildlife and tropical scenery.\n\nPopular destinations:\n1. Kochi.\n2. Munnar.\n3. Alleppey.\n4. Varkala.\n5. Wayanad.\n6. Thekkady.\n7. Kovalam.\n\nRecommended duration: 4 to 7 days.\n\nBudget:\nBudget travelers can spend approximately ₹1,000 to ₹2,000 per day excluding accommodation.\nMid-range travelers can spend approximately ₹2,500 to ₹5,000 per day excluding accommodation.\n\nFood:\nAppam, puttu, Kerala parotta, fish curry, sadya and seafood are popular.\n\nBest time:\nOctober to March is generally popular for many destinations.\n\nTransportation:\nBuses, trains, taxis and boats are available depending on the destination.\n\nSuggested itinerary:\nDay 1: Kochi sightseeing.\nDay 2: Travel to Munnar.\nDay 3: Munnar sightseeing.\nDay 4: Travel toward Alleppey.\nDay 5: Alleppey backwater experience.\n', 'Hyderabad': 'Destination: Hyderabad, Telangana, India\n\nHyderabad is famous for its history, architecture and food.\n\nPopular attractions:\n1. Charminar.\n2. Golconda Fort.\n3. Hussain Sagar Lake.\n4. Salar Jung Museum.\n5. Chowmahalla Palace.\n6. Ramoji Film City.\n7. Birla Mandir.\n\nRecommended duration: 2 to 3 days.\n\nBudget:\nBudget travelers can spend approximately ₹800 to ₹1,500 per day excluding accommodation.\n\nFood:\nHyderabadi biryani, haleem, kebabs and Irani chai are popular.\n\nBest time:\nOctober to February is generally comfortable.\n\nTransportation:\nMetro, buses, taxis and auto-rickshaws are available.\n\nSuggested itinerary:\nDay 1: Charminar, Chowmahalla Palace and local markets.\nDay 2: Golconda Fort, Salar Jung Museum and Hussain Sagar.\nDay 3: Ramoji Film City.\n'}

print('Destinations loaded:', len(travel_data))
print(list(travel_data.keys()))

## 4. Convert travel data into documents

In [ ]:
documents = []

for destination, content in travel_data.items():
    documents.append(
        Document(
            page_content=content,
            metadata={
                "destination": destination,
                "source": f"{destination}_travel_guide"
            }
        )
    )

print("Documents created:", len(documents))


## 5. Split documents into chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))


## 6. Create embeddings

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")


In [ ]:
chunk_texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)


## 7. Create FAISS vector database

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

print("FAISS vector database created!")
print("Number of vectors:", index.ntotal)


## 8. Create the retriever

In [ ]:
def retrieve_documents(query, k=4):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        if idx < len(chunks):
            results.append({
                "document": chunks[idx],
                "distance": float(distance)
            })

    return results


## 9. Test retrieval

In [ ]:
query = "What are the best places to visit in Chennai?"
results = retrieve_documents(query)

for i, result in enumerate(results):
    print("\n==============================")
    print("RESULT", i + 1)
    print("==============================")
    print("Destination:", result["document"].metadata["destination"])
    print(result["document"].page_content)


## 10. Load the language model

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300
)

print("TravelBuddy language model loaded!")


In [ ]:
# If the previous cell causes a memory error, run this cell instead.

# model_name = "google/flan-t5-small"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
# generator = pipeline(
#     "text2text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=250
# )
# print("Small model loaded!")


## 11. Build the RAG system

In [ ]:
def create_context(results):
    context_parts = []

    for result in results:
        destination = result["document"].metadata["destination"]
        content = result["document"].page_content
        context_parts.append(
            f"DESTINATION: {destination}\n{content}"
        )

    return "\n\n".join(context_parts)


def travelbuddy_rag(question, k=4):
    retrieved = retrieve_documents(question, k)
    context = create_context(retrieved)

    prompt = f"""
You are TravelBuddy, an AI travel planning assistant.

Answer the user's question using ONLY the travel information
provided in the context.

If the information is not available in the context,
say that the information is not available in the TravelBuddy
knowledge base.

Do not invent attractions, prices, timings or facts.

Give a clear and useful answer.

Context:
{context}

User Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        do_sample=False
    )[0]["generated_text"]

    sources = []
    for result in retrieved:
        source = result["document"].metadata["destination"]
        if source not in sources:
            sources.append(source)

    return response, sources


## 12. Test the complete RAG

In [ ]:
question = "Plan a 3 day trip to Chennai."

answer, sources = travelbuddy_rag(question)

print("TRAVELBUDDY")
print("=" * 50)
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)


## 13. Trip planner

In [ ]:
def plan_trip(destination, days, budget, travel_style):
    question = f"""
Create a {days}-day travel itinerary for {destination}.

Budget: {budget}
Travel style: {travel_style}

Include:
- Places to visit
- Food suggestions
- Daily plan
- Transportation suggestions
- Budget-friendly advice

Use only information available in the TravelBuddy knowledge base.
"""

    answer, sources = travelbuddy_rag(question)
    return answer, sources


answer, sources = plan_trip(
    "Chennai",
    3,
    "Budget",
    "Sightseeing and food"
)

print(answer)
print("\nSources:", ", ".join(sources))


## 14. Destination recommender

In [ ]:
def recommend_destination(preference):
    question = f"""
Recommend the most suitable destination from the TravelBuddy
knowledge base for someone who wants:

{preference}

Compare the relevant destinations and explain why.
Mention attractions, food and approximate budget.
Do not invent information.
"""

    answer, sources = travelbuddy_rag(question, k=6)
    return answer, sources


answer, sources = recommend_destination(
    "beaches, food and a relaxing vacation"
)

print(answer)
print("\nSources:", ", ".join(sources))


## 15. Gradio chatbot

In [ ]:
import gradio as gr

def chatbot_response(message, history):
    answer, sources = travelbuddy_rag(message)
    source_text = "\n\nSources: " + ", ".join(sources)
    return answer + source_text

demo = gr.ChatInterface(
    fn=chatbot_response,
    title="TravelBuddy RAG",
    description="AI Travel Assistant powered by Retrieval-Augmented Generation",
    examples=[
        "Plan a 3 day trip to Chennai",
        "What places should I visit in Goa?",
        "Suggest a budget trip to Jaipur",
        "What food should I try in Delhi?",
        "Give me a 2 day Mumbai itinerary",
        "Which destination is good for beaches?"
    ]
)

demo.launch()


## 16. Project notes

This notebook uses a static demo travel knowledge base. Prices, opening hours, transport schedules and availability are not live data. For a production travel assistant, connect verified live APIs or current web sources.

### RAG pipeline
1. Travel documents are loaded.
2. Documents are split into chunks.
3. Chunks are converted into embeddings.
4. Embeddings are stored in FAISS.
5. A user query is embedded and relevant chunks are retrieved.
6. Retrieved context is passed to FLAN-T5.
7. The model generates a grounded answer with sources.